In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_parquet('../data/raw/yellow_tripdata_2026-01.parquet')
df.head(3)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75


### Data Hygiene & Outlier

In [4]:
initial_rows = len(df)

df_clean = df[
    (df['trip_distance'] > 0) & (df['trip_distance'] < 50) &
    (df['fare_amount'] >= 2.50) & (df['fare_amount'] <= 200) &
    (df['passenger_count'] > 0)
].copy()
cleaned_rows = len(df_clean)
dropped_rows = initial_rows - cleaned_rows
percentage = round((dropped_rows/initial_rows)*100, 2)

print(f'Number of rows dropped {dropped_rows} which represent {percentage}% of the dataset')

Number of rows dropped 1174310 which represent 31.53% of the dataset


### Time-Series & Demand Forecasting

In [5]:
# Extract the hour of the day (0–23) and the day of the week from the pickup datetime.
df_clean['pickup_hour'] = df_clean['tpep_pickup_datetime'].dt.hour
df_clean['pickup_day'] = df_clean['tpep_pickup_datetime'].dt.day_name()

# Group the data to find the average trip volume and average fare amount for every hour of the week.
hourly_stats = df_clean.groupby(['pickup_hour', 'pickup_day']).agg(
    avg_fare = ('fare_amount', 'mean'),
    trip_count = ('fare_amount', 'count')
).reset_index()

# Which specific hour and day of the week yields the highest average fare?
highest_avg_fare = hourly_stats.loc[hourly_stats['avg_fare'].idxmax(), ['pickup_hour', 'pickup_day', 'avg_fare', 'trip_count']]

highest_avg_fare

pickup_hour            3
pickup_day       Tuesday
avg_fare       35.322496
trip_count           617
Name: 26, dtype: object

In [6]:
hourly_stats.to_parquet('../data/processed/hourly_stats.parquete', index = False)

### The "Generous Rider" Metric (Feature Engineering)

In [7]:
#Filter the dataset to include only credit card payments (payment_type == 1)
df_credit_card = (df_clean[
    (df_clean['payment_type'] == 1) &
    (df_clean['fare_amount'] > 0)
]).copy()

# Create a new feature column named tip_percentage, calculated as:$$\text{tip\_percentage} = \left( \frac{\text{tip\_amount}}{\text{fare\_amount}} \right) \times 100$$
df_credit_card['tip_percentage'] = round((df_clean['tip_amount'] / df_clean['fare_amount'])*100, 2)
df_credit_card

# Find the top 5 pickup locations (PULocationID) that have the highest average tip_percentage (exclude locations with fewer than 100 total trips to avoid statistical flukes).
df_pickup = df_credit_card.groupby('PULocationID').agg(
    avg_tip_pct = ('tip_percentage', 'mean'),
    trips = ('tip_percentage', 'count')
)
df_pickup

top_5_tip_locations = df_pickup[df_pickup['trips'] > 100].sort_values(by = 'avg_tip_pct', ascending = False).head()
print(f'Top 5 pickup locations by average tipping percentage')
print(top_5_tip_locations)

Top 5 pickup locations by average tipping percentage
              avg_tip_pct   trips
PULocationID                     
237             27.710136  121965
142             27.672274   77953
239             27.670256   64061
90              27.627326   33600
113             27.545958   30261


In [8]:
# saving the dataframe as a new dataset
df_credit_card.to_parquet('../data/processed/cleaned_credit_card_trips.parquet', index = False)